# Week 13: Creating Text Embedding Models

*One class this week (Thanksgiving).*

Week 12 ended with one diagnosis: *Qwen knows SQL, but it does not know this database.* Plain Qwen dropped conditions, wrote people's words as table values (`'Pro 2'`, `'on the way'`) and picked the wrong columns. We fixed that by training Qwen with LoRA.

Today we fix the same problem a second way, without touching Qwen: we **show it the right example**. For each new question we find the solved Week 12 pair whose SQL has the same structure and paste it into the prompt. If the example's SQL says `status = 'returned'` for "sent back", Qwen can copy the convention instead of guessing it.

To find that example we need a retriever, and MiniLM as downloaded (Weeks 6 and 9) has the wrong idea of "similar". So today we teach MiniLM a new meaning of "similar".

That is what creating a text embedding model usually means in practice. You rarely start from random weights. You take a pretrained encoder and fine-tune it on pairs of texts that should land close together. MiniLM itself was made this way: a pretrained transformer fine-tuned with today's loss on about a billion sentence pairs.

| Step | What we do |
|---|---|
| 1 | Load the Week 12 question/SQL pairs and give each question its SQL shape |
| 2 | Base MiniLM: does the nearest past question have the right shape? |
| 3 | Count how often it finds the right example for the 46 test questions |
| 4 | Fine-tune MiniLM with a contrastive loss |
| 5 | Measure again, then hand the retrieved pair to Qwen |

A GPU (Colab T4) makes training take seconds. It also runs on CPU, just slower.

## Why "similar" depends on the task

In Week 7 we saw that one solved example in the prompt (few-shot) helps a small model. We have 381 solved question/SQL pairs from Week 12. For a new question, which one should we show Qwen? The one whose SQL has the same structure, so Qwen only has to swap in the new values.

| Question | SQL |
|---|---|
| Show me everything Chloe has purchased. | `SELECT product FROM orders WHERE customer = 'Chloe'` |
| On which day did Chloe purchase the AeroBeat Pro 2? | `SELECT order_date FROM orders WHERE customer = 'Chloe' AND product = 'AeroBeat Pro 2'` |
| Which products has Sam bought? | `SELECT product FROM orders WHERE customer = 'Sam'` |

For a topic search, rows 1 and 2 are the close pair: same person, same verb. For writing SQL, rows 1 and 3 are the pair that matters: the same query with a different name. A general embedding model learned the first kind of similarity. Nothing in it knows we care about the second.

To label the second kind, we replace every value in the SQL with `?`. We call the result the question's **SQL shape**. Rows 1 and 3 both have the shape `SELECT product FROM orders WHERE customer = ?`.

In [ ]:
import re, random
import pandas as pd

DATA = "https://raw.githubusercontent.com/razi-cmu/CPS535/main/data/"

def load_pairs(source, question_col="question", sql_col="sql"):   # Week 12, unchanged
    """Load prompt/response pairs from any CSV (swap in your own project's file here)."""
    df = pd.read_csv(source)
    return [{"question": q, "sql": s} for q, s in zip(df[question_col], df[sql_col])]

TRAIN = load_pairs(DATA + "aerobeat_sql_train.csv")
TEST  = load_pairs(DATA + "aerobeat_sql_test.csv")

def shape(sql):
    sql = re.sub(r"'[^']*'", "?", sql)     # text values -> ?
    return re.sub(r"\b\d+\b", "?", sql)    # numbers -> ?

train_q      = [p["question"] for p in TRAIN]
train_shapes = [shape(p["sql"]) for p in TRAIN]
SHAPES = sorted(set(train_shapes))

print(len(TRAIN), "training pairs,", len(TEST), "test pairs,", len(SHAPES), "shapes\n")
for sh in SHAPES:
    print(f"{train_shapes.count(sh):3d}  {sh}")

**What the output tells you.** The 381 training questions fall into 10 shapes. Recall how Week 12 built the data: each shape is asked with 3 wordings in training, and the test set uses a 4th wording that never appears in training. So a test question cannot be matched by copying words. The model has to recognize what kind of question it is.

## Base MiniLM as a retriever

We load MiniLM with `transformers` and pool it ourselves, so we can train it later. `mean_pool()` is Week 11's, unchanged. Each vector is normalized to length 1, so a dot product is the cosine (Week 9).

`encode()` keeps gradients (we need them for training). `embed()` is the same thing with gradients off, for searching. `show()` prints the 3 nearest training questions for one question and marks whether each has the right shape.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
tok  = AutoTokenizer.from_pretrained(MODEL_NAME)
base = AutoModel.from_pretrained(MODEL_NAME).to(device)

def mean_pool(hidden, mask):                         # Week 11
    mask = mask.unsqueeze(-1).float()
    return (hidden * mask).sum(1) / mask.sum(1)

def encode(model, texts):
    batch  = tok(texts, padding=True, truncation=True, return_tensors="pt").to(device)
    hidden = model(**batch).last_hidden_state
    return F.normalize(mean_pool(hidden, batch["attention_mask"]), dim=-1)

@torch.no_grad()
def embed(model, texts):
    model.eval()
    return encode(model, texts)

def show(model, question, gold_sql=None, k=3):
    scores = (embed(model, [question]) @ embed(model, train_q).T)[0]
    best = scores.topk(k)
    print("Question:", question)
    for s, j in zip(best.values.tolist(), best.indices.tolist()):
        mark = "" if gold_sql is None else ("same shape " if train_shapes[j] == shape(gold_sql) else "WRONG shape")
        print(f"  {s:.3f}  {mark}  {train_q[j]}\n                      {TRAIN[j]['sql']}")

QUERY, QUERY_SQL = "Show me everything Chloe has purchased.", "SELECT product FROM orders WHERE customer = 'Chloe'"
show(base, QUERY, QUERY_SQL)

**What the output tells you.** Chloe appears in the training questions, but only in other shapes (how many orders she placed, when she bought something). Look at the top 3: do they share the shape, or just the name "Chloe" and the word "purchase"? A general model weighs shared names and words heavily, because for most searches that is exactly what similar means. As in Week 9, compare ranks, not raw scores.

One question is not a measurement. `right_example()` repeats the check for all 46 test questions: for each one it takes only the single nearest training question (that is the example Qwen would see) and records whether its shape matches. The score is a count, and the cell lists the test questions it missed so you can read them.

This is the retrieval version of Week 12's execution accuracy. We cannot run the retrieved example's SQL instead: for the Chloe question the right example is about Sam, so its rows would never match. Execution accuracy comes back once Qwen writes the query (Try-it 4).

In [ ]:
def right_example(model):
    index = embed(model, train_q)                                        # 381 x 384
    best  = (embed(model, [p["question"] for p in TEST]) @ index.T).argmax(dim=1)   # nearest only
    return [train_shapes[j] == shape(p["sql"]) for j, p in zip(best.tolist(), TEST)]

before = right_example(base)
print(f"base MiniLM: right example for {sum(before)}/{len(TEST)} test questions\n")
for p, ok in zip(TEST, before):
    if not ok:
        print("  missed:", p["question"])

Write the count on the board, and look at which kinds of question were missed. We will compare after training.

## Teaching MiniLM our meaning of similar

We need two things: pairs that should be close, and texts that should be far apart.

- **Positive pair:** two training questions with the same shape, e.g. "Which products has Sam bought?" and "List the products ordered by Sofia."
- **Negatives:** we get them for free. In a batch of 10 pairs, every other pair's question is a negative. These are called **in-batch negatives**.

The loss is Week 10's CLIP loss with text on both sides. Recap in one line: build the cosine matrix between anchors and positives, multiply by a scale, softmax each row, and the loss is minus the log of the diagonal entry. We use a scale of 20 (the `sentence-transformers` default; CLIP learned 100).

A batch of 3, as an example. Each row should pick its own column:

| anchor \ positive | "List the products ordered by Sofia." | "Count Leo's orders." | "Has order 1089 arrived yet?" |
|---|---|---|---|
| "Which products has Sam bought?" | pull together | push apart | push apart |
| "How many orders has Fatima placed?" | push apart | pull together | push apart |
| "What is the status of order 1015?" | push apart | push apart | pull together |

**Why one pair per shape in each batch.** If two rows in a batch had the same shape, the loss would push apart two questions that should be close (a **false negative**). With 10 shapes, a batch of 10 rows with one row per shape makes every off-diagonal entry a true negative. A model that cannot tell the shapes apart would score ln 10 = 2.30, so watch the loss fall below that.

We train a fresh copy and keep `base` for the comparison. The loop is Week 4's: forward, loss, backward, step. The learning rate is small (2e-5, as in Week 11) because we are adjusting a pretrained model, not training one.

In [ ]:
model = AutoModel.from_pretrained(MODEL_NAME).to(device)      # fresh copy to fine-tune
by_shape = {sh: [q for q, t in zip(train_q, train_shapes) if t == sh] for sh in SHAPES}

def make_batch():
    anchors, positives = [], []
    for sh in SHAPES:                                   # one pair per shape
        a, p = random.sample(by_shape[sh], 2)
        anchors.append(a); positives.append(p)
    return anchors, positives

def contrastive_loss(a, p, scale=20.0):
    logits = scale * (a @ p.T)                          # 10 x 10 cosine matrix
    labels = torch.arange(len(a), device=a.device)      # right answer is the diagonal
    return F.cross_entropy(logits, labels)

random.seed(0); torch.manual_seed(0)
opt = torch.optim.AdamW(model.parameters(), lr=2e-5)
model.train()
for step in range(1, 301):
    anchors, positives = make_batch()
    loss = contrastive_loss(encode(model, anchors), encode(model, positives))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0:
        print(f"step {step:3d}  loss {loss.item():.3f}")

300 steps of 20 questions is 6,000 questions, about 16 passes over the 381 training questions. The loss is noisy from step to step because every batch is a new random draw.

## Did it work?

Same check on the same 46 questions. The list shows what changed: questions that went from missed to right, any that went the other way, and any still missed. Then the Chloe question again, and one of your own. The last lines turn the top match into a few-shot example for Week 12's Qwen.

In [ ]:
after = right_example(model)
print(f"right example:  before {sum(before)}/{len(TEST)}   after {sum(after)}/{len(TEST)}\n")
for p, b, a in zip(TEST, before, after):
    if a and not b:  print("  fixed:       ", p["question"])
    if b and not a:  print("  now missed:  ", p["question"])
    if not (a or b): print("  still missed:", p["question"])
print()
show(model, QUERY, QUERY_SQL)
print()
show(model, "Which products did Chloe buy?")          # try your own question

def example_for(model, question):
    j = (embed(model, [question]) @ embed(model, train_q).T)[0].argmax().item()
    p = TRAIN[j]
    return f"Here is a similar question and its SQL:\nQuestion: {p['question']}\nSQL: {p['sql']}"

print("\n" + example_for(model, QUERY))

**What the output tells you.**
- Training pulls same-shape questions together and pushes different shapes apart, so names, products and cities count for less than the kind of question.
- The test wordings never appeared in training. A right example therefore means the model recognized the shape, not that it memorized words.
- Nothing about MiniLM's architecture changed. Same 6 layers, same 384 numbers per question. Only the weights moved, and with them the meaning of "close". As in Week 11, fine-tuning learns what the labels contain: our labels said "same SQL shape", so that became similar.

**From retrieval to Qwen.** The last printout is the piece that connects Weeks 9, 12 and 13. Week 9 retrieved documents and pasted them into the prompt. Here we retrieve a solved example instead and paste it into Week 12's `sql_prompt()`, between the question and the table. Qwen then gets a working query of the right shape to adapt, including this database's conventions, without any fine-tuning of Qwen itself. That is Week 12's problem solved by prompting instead of training.

This only works when the pool holds a solved example of the right kind. A question outside the 10 shapes (one that needs `>`, say) still gets the nearest wrong example. The fix is cheap: write one solved pair of the new kind and add it to `TRAIN`; MiniLM can retrieve it right away and Qwen needs no retraining.

**Why not a 10-label classifier?** With exactly 10 shapes, a Week 11 classifier would also work. An embedding model is more flexible: to support a new kind of question, you add one solved pair to the index and it can be retrieved right away, with no retraining. The same model also serves search, clustering (Week 6) and RAG (Week 9).

## What can still go wrong

| Problem | Why | Where it goes |
|---|---|---|
| A question no shape covers ("Which city orders the most?") still gets a nearest match | Retrieval always returns something | Try-it 1 |
| Qwen copies the example's values (Sam instead of Chloe) | Small models copy examples (Weeks 7 and 10) | Say in the instruction that values come from the question; Try-it 4 |
| Worse at general topic search | It was trained only on our meaning of similar | Try-it 2 |
| A wrong or malicious pair in the index gets copied into answers | Whatever is retrieved is trusted | Week 14 |
| The count looks better than real use | 46 questions, one new wording per shape; real users phrase things in many more ways | Collect real questions |

## Try it

1. **Out-of-scope questions.** Run `show(model, "Which city orders the most?")` and `show(model, "What is the weather in Flint?")`. Compare their top scores with in-scope questions. Pick a score threshold below which `example_for()` should return nothing.
2. **Forgetting.** Take Week 9's `DOCS` and search "Can I send them back if I don't like them?" with `base` and with `model`. Does the fine-tuned model still find the Returns article?
3. **False negatives.** Replace `make_batch()` with 10 random pairs from any shapes (so two rows can share a shape). Retrain and compare the count.
4. **Close the loop.** In the Week 12 notebook, give `sql_prompt()` an optional `example` argument placed between the question and the table, pass `example_for(model, question)`, and rerun `execution_accuracy()` on the 46 test questions with the base Qwen. Compare with Week 12's zero-shot and LoRA numbers.
5. **Too fast.** Set `lr=1e-3` and retrain. What happens to the loss and to the count?

## Adapting this to your own project

The recipe is the same for any notion of similar:

1. Say in one sentence what "similar" means for your task.
2. Collect pairs that should be close.
3. Build batches where no two rows belong together (one row per group, when you have groups).
4. Train with the in-batch loss, a small learning rate (2e-5) and a few hundred steps.
5. On held-out queries, count how often retrieval is right, before and after. If it does not improve, keep the base model.

| Task | Positive pair | What to count |
|---|---|---|
| FAQ or help-desk matching | user question, FAQ question with the same answer | nearest FAQ is the right one |
| RAG over your own documents (Week 9) | question, chunk that answers it | right chunk among the top 3 (all of them go into the prompt) |
| Duplicate detection | two reports of the same issue | nearest report is a true duplicate |
| Few-shot example picker (today) | question, question with the same output structure | nearest question has the same shape |

For a (question, chunk) task, use a different chunk in each row of a batch; the code above does not change otherwise.

**The library version.** `sentence-transformers` packages exactly the loss we wrote, under the name `MultipleNegativesRankingLoss` (scale 20 by default):

```python
from sentence_transformers import SentenceTransformer, losses
st   = SentenceTransformer(MODEL_NAME)
loss = losses.MultipleNegativesRankingLoss(st)   # train it with SentenceTransformerTrainer
```

**Next week:** we test our own systems for bias, hallucination and prompt injection, including what happens when a retrieved document or example carries instructions.